EDA for all final dataset 

In [1]:
import pandas as pd
import numpy as np

In [2]:
#load data 
df = pd.read_csv("../../data/final_datasets/feature_matrix_all_dimensions.csv", dtype={"plr_id": str})

#Missings per PLR - are values missing im many PLRs or are they concentrated on some PLRs? 
num_cols = df.select_dtypes(include=[np.number]).columns
nan_per_plr = df[num_cols].isna().sum(axis=1)

print("NaNs per PLR – Dispersion:")
print(nan_per_plr.value_counts().sort_index())

missing = (
    df.loc[nan_per_plr > 0, ["plr_id", "plr_name", "bez"]]
    .assign(n_nan=nan_per_plr[nan_per_plr > 0])
    .sort_values("n_nan", ascending=False)
)
print(f"\n{len(missing)} von {len(df)} PLR with at least one missing value:")
print(missing.to_string(index=False))



NaNs per PLR – Dispersion:
0     499
1      27
2       5
3       4
5       1
8       2
9       1
12      1
28      2
Name: count, dtype: int64

43 von 542 PLR with at least one missing value:
  plr_id                          plr_name                             bez  n_nan
06200418                           Landweg        06 - Steglitz-Zehlendorf     28
03400831                      Pankower Tor                     03 - Pankow     28
05300838                        Gartenfeld                    05 - Spandau     12
10100205 Gewerbegebiet Bitterfelder Straße        10 - Marzahn-Hellersdorf      9
03300515               Blankenburger Süden                     03 - Pankow      8
04400725            Güterbahnhof Grunewald 04 - Charlottenburg-Wilmersdorf      8
12200411               Schumacher-Quartier              12 - Reinickendorf      5
07200412                Schöneberger Linse       07 - Tempelhof-Schöneberg      3
08200728                        Ortolanweg                   08 - Neuk

In [3]:
#are the missings in one column?
mask1 = nan_per_plr == 1
einzel = df.loc[mask1, num_cols]
spalte_je_plr = einzel.isna().idxmax(axis=1)
print("which column is missing in the 1-NaN-PLR:")
print(spalte_je_plr.value_counts())

#Do the heavily affected PLR coincide with the non-residential flag?
heavy = df.loc[nan_per_plr >= 5, ["plr_id", "plr_name", "re_miet_dimension_anwendbar"]]
print("\nHeavily affected PLR vs. residential-applicability flag:")
print(heavy.to_string(index=False))


which column is missing in the 1-NaN-PLR:
soc_median_income_2023    25
re_miete_trend             2
Name: count, dtype: int64

Heavily affected PLR vs. residential-applicability flag:
  plr_id                          plr_name  re_miet_dimension_anwendbar
03300515               Blankenburger Süden                            0
03400831                      Pankower Tor                            0
04400725            Güterbahnhof Grunewald                            1
05300838                        Gartenfeld                            0
06200418                           Landweg                            0
10100205 Gewerbegebiet Bitterfelder Straße                            0
12200411               Schumacher-Quartier                            1


Decision: include Güterbhf Grunewald and Schumacher Quartier also in the flag re_miet_dimension_anwendbar 

In [4]:
fix_ids = ["04400725", "12200411"]  # Güterbahnhof Grunewald, Schumacher-Quartier
df.loc[df["plr_id"].isin(fix_ids), "re_miet_dimension_anwendbar"] = 0

#Result of inspecting the NaNs

Missing values are highly concentrated, not dispersed: 499 of 542 PLR are complete, and only 43 carry any NaN. The most affected — Landweg and Pankower Tor (28 each), plus Gartenfeld, Gewerbegebiet Bitterfelder Straße, Blankenburger Süden, Güterbahnhof Grunewald, and Schumacher-Quartier — are development sites, former railway/industrial land, and barely-populated quarters that legitimately lack structure across all three dimensions. The long single-NaN tail is a column property, not a PLR one: 25 of 27 cases trace to soc_median_income_2023. 

The gaps are therefore structural, calling for imputation of the income column and consistent cross-dimension flagging of the non-residential sites — though two heavily affected quarters (Güterbahnhof Grunewald, Schumacher-Quartier) carry re_miet_dimension_anwendbar == 1, showing the real-estate flag alone does not capture them. 


Decisions: 

The soc_income_dimension variable doesn't go into the modelling. We decide to drop the non-residential PLRs with more than 5 missings for the modelling stage. 

Those are the non-residential PLRs that will be dropped for modelling: Landweg, Pankower Tor, Gartenfeld, Gewerbegebiet Bitterfelder Straße, Blankenburger Süden, Güterbahnhof, Schumacher-Quartier. 

Now let's inspect the missings without those PLRs, where is only one feature missing in a PLRs. 

In [5]:
# --- define feature columns for the missing-count (exclude id/label/flag) ---
non_feature = ["plr_id", "plr_name", "bez", "re_miet_dimension_anwendbar"]
feature_cols = [c for c in df.columns if c not in non_feature]

# --- drop the 7 non-residential PLR (>= 5 missings) ---
nan_per_plr = df[feature_cols].isna().sum(axis=1)
df_model = df.loc[nan_per_plr < 5].copy()

# --- drop descriptive income + applicability flag ---
df_model = df_model.drop(columns=["soc_median_income_2023", "re_miet_dimension_anwendbar"])

print(f"Modelling set: {len(df_model)} PLR, {df_model.shape[1]} columns")

Modelling set: 535 PLR, 56 columns


In [6]:
#inspect remaining missingness
n = len(df_model)
miss = df_model.isna().sum()
overview = pd.DataFrame({
    "n_missing": miss,
    "pct": (miss / n * 100).round(1),
})
overview = overview[overview["n_missing"] > 0].sort_values("n_missing", ascending=False)
print(overview)

                              n_missing  pct
re_miete_trend                        9  1.7
re_miete_niveau                       5  0.9
re_brw_niveau                         1  0.2
re_brw_trend                          1  0.2
re_cov_wohnen_2025                    1  0.2
com_upscale_share_level_2026          1  0.2
com_upscale_share_slope               1  0.2


In [7]:
rent_cols = ["re_miete_niveau", "re_miete_trend"]
brw_cols  = ["re_brw_niveau", "re_brw_trend", "re_cov_wohnen_2025"]
all_gap   = rent_cols + brw_cols

#which block is missing for each affected PLR
affected = df_model[df_model[all_gap].isna().any(axis=1)].copy()

def missing_block(row):
    rent_missing = row[rent_cols].isna().any()
    brw_missing  = row[brw_cols].isna().any()
    if rent_missing and brw_missing: return "rent+brw"
    if rent_missing:                 return "rent only"
    if brw_missing:                  return "brw only"
    return "-"

affected["missing_block"] = affected.apply(missing_block, axis=1)

print(affected[["plr_id", "plr_name",
                "re_miete_niveau", "re_miete_trend",
                "re_brw_niveau", "re_cov_wohnen_2025",
                "re_miete_unsicher", "missing_block"]].to_string(index=False))

  plr_id           plr_name  re_miete_niveau  re_miete_trend  re_brw_niveau  re_cov_wohnen_2025  re_miete_unsicher missing_block
03300413          Karow Ost              NaN             NaN     500.022367            0.959013                  1     rent only
03300517        Märchenland              NaN             NaN     285.571015            0.115828                  1     rent only
04200207           Eichkamp            16.67             NaN     996.036121            0.034252                  1     rent only
05200419       Am Heideberg              NaN             NaN     469.435055            0.928344                  1     rent only
07200412 Schöneberger Linse            22.60           0.854            NaN                 NaN                  0      brw only
08200728         Ortolanweg              NaN             NaN     600.000000            0.084402                  1     rent only
08200729     Britzer Garten              NaN             NaN     625.333179            0.109536  

Decision: Domain Knowledge on Schöneberger Linse: doesn't have a BRW because it was not a residential area in 2022. However, now it is. Therefore best to impute KNN on BRW. 

In [8]:
#two hypotheses why the rent data is missing: Neubaugebiete or non-residential? 

#bring fallzahl_mittel in from the angebotsmieten source (not in the merged matrix) ---
src = pd.read_csv("../../data/real_estate/final_variables/var1_2_angebotsmieten.csv", dtype={"plr_id": str})  # adjust path
src["plr_id"] = src["plr_id"].str.zfill(8)
df_check = df_model.merge(src[["plr_id", "fallzahl_mittel"]], on="plr_id", how="left")

#neubau share (or reuse re_neubau_share if already derived)
df_check["neubau_share"] = df_check["re_anteil_y2001_2010"] + df_check["re_anteil_y2011_plus"]

#therefore: two missingness groups
group_a = ["03300413", "03300517", "05200419", "08200728", "08200729"]  # fallzahl NaN: no offers -> niveau+trend missing
group_b = ["04200207", "09100306", "11300723", "12200414"]              # fallzahl 3-4: niveau present, trend missing

cols = ["plr_id", "plr_name", "fallzahl_mittel",
        "re_cov_wohnen_2025", "neubau_share", "re_anteil_vor_1919"]

print("GROUP A — no rental offers (fallzahl NaN):")
print(df_check[df_check["plr_id"].isin(group_a)][cols].round(3).to_string(index=False))
print("\nGROUP B — too few offers for a trend (fallzahl 3–4):")
print(df_check[df_check["plr_id"].isin(group_b)][cols].round(3).to_string(index=False))

GROUP A — no rental offers (fallzahl NaN):
  plr_id       plr_name  fallzahl_mittel  re_cov_wohnen_2025  neubau_share  re_anteil_vor_1919
03300413      Karow Ost              NaN               0.959         0.270               0.022
03300517    Märchenland              NaN               0.116         0.031               0.006
05200419   Am Heideberg              NaN               0.928         0.000               0.754
08200728     Ortolanweg              NaN               0.084         0.000               0.009
08200729 Britzer Garten              NaN               0.110         0.179               0.017

GROUP B — too few offers for a trend (fallzahl 3–4):
  plr_id      plr_name  fallzahl_mittel  re_cov_wohnen_2025  neubau_share  re_anteil_vor_1919
04200207      Eichkamp              3.0               0.034         0.021               0.023
09100306   Späthsfelde              3.0               0.242         0.141               0.027
11300723 Bornitzstraße              3.0            

None of the hypotheses are true:

Group A: Karow Ost (cov 0.96) and Am Heideberg (cov 0.93, 75% Altbau) are strongly residential but owner-occupied areas that don't generate rental listings. Märchenland/Ortolanweg/Britzer Garten are low-coverage green-edge areas. 

Group B: only Bornitzstraße is newly-built (neubau 0.92); Eichkamp/Späthsfelde/TXL are not new-build at all. So "Neubaugebiete" doesn't hold as a class either.

Small caveat: building age is based on the Zensus 2022 Gebäude- und Wohnungszählung (Stichtag 15 May 2022) and therefore reflects the building stock as of that date. Construction completed after May 2022 is not captured, so PLR with major post-2022 development (e.g. the Schumacher-Quartier redevelopment) will under-represent their actual Neubau share.

Domain knowledge comes in here: Tegel 2026 is neither a meaningful rental market nor a residential Kiez and therefore is also to be excluded. 


Decisions:

9 PLR with rent missing (niveau and/or trend) → KNN imputes the rent cells from BRW + coverage + social + commercial.
Schöneberger Linse with BRW/coverage missing → KNN imputes those cells from rent + the rest.

Recoding Baualter Variable as it sums up to 1 and can't be used for clustering 

In [9]:
#does it sum up to 1? (round issues most likely)

ba = ["re_anteil_vor_1919", "re_anteil_y1919_1948", "re_anteil_y1949_1978",
      "re_anteil_y1979_1990", "re_anteil_y1991_2000", "re_anteil_y2001_2010",
      "re_anteil_y2011_plus"]

ba_sum = df_model[ba].sum(axis=1)

print("Baualter row-sum check on the modelling set:")
print(ba_sum.describe().round(4).to_string())

#how far do they drift from 1?
dev = (ba_sum - 1).abs()
print(f"\nMax deviation from 1: {dev.max():.4f}")
print(f"PLR deviating > 1%:  {(dev > 0.01).sum()}")
print(f"PLR deviating > 2%:  {(dev > 0.02).sum()}")

#show the ones that drift, for the record
drifters = df_model.loc[dev > 0.01, ["plr_id", "plr_name"]].assign(ba_sum=ba_sum[dev > 0.01].round(4))
print("\nPLR with sum deviating > 1% (rounding in source shares, retained as-is):")
print(drifters.to_string(index=False))

Baualter row-sum check on the modelling set:
count    535.0000
mean       0.9998
std        0.0024
min        0.9804
25%        0.9992
50%        1.0000
75%        1.0007
max        1.0256

Max deviation from 1: 0.0256
PLR deviating > 1%:  5
PLR deviating > 2%:  1

PLR with sum deviating > 1% (rounding in source shares, retained as-is):
  plr_id               plr_name  ba_sum
03300517            Märchenland  0.9826
05300840        Nonnendammallee  1.0256
08200728             Ortolanweg  0.9840
09401432 Siedlung Kämmereiheide  0.9864
11300723          Bornitzstraße  0.9804


In [10]:
#derive two disjoint Baualter features
df_model["re_altbau_share"] = df_model["re_anteil_vor_1919"]                       # pre-1919 Gründerzeit
df_model["re_neubau_share"] = (
    df_model["re_anteil_y2001_2010"] + df_model["re_anteil_y2011_plus"]            # 2001+
)

#energetic-modernisation potential is the complement of Neubau 
#However: NOT stored as a feature (near-constant + overlaps Altbau): 1 - re_neubau_share

#drop the 7 original bands (collapsed into the two shares above)
baualter_bands = [
    "re_anteil_vor_1919", "re_anteil_y1919_1948", "re_anteil_y1949_1978",
    "re_anteil_y1979_1990", "re_anteil_y1991_2000",
    "re_anteil_y2001_2010", "re_anteil_y2011_plus",
]
df_model = df_model.drop(columns=baualter_bands)

#verify the two new shares aren't redundant with each other or the market
checks = ["re_altbau_share", "re_neubau_share", "re_miete_niveau", "re_brw_niveau"]
print(df_model[checks].corr(method="spearman").round(2).to_string())
print(f"\nColumns now: {df_model.shape[1]}")

                 re_altbau_share  re_neubau_share  re_miete_niveau  re_brw_niveau
re_altbau_share             1.00            -0.04             0.61           0.72
re_neubau_share            -0.04             1.00             0.30          -0.10
re_miete_niveau             0.61             0.30             1.00           0.62
re_brw_niveau               0.72            -0.10             0.62           1.00

Columns now: 51


In [11]:
#Are any PLR still flagged? (check only flags still present in df_model)
for flag in ["re_baualter_unsicher", "re_brw_trend_outlier", "re_miet_dimension_anwendbar"]:
    if flag not in df_model.columns:
        print(f"{flag}: already dropped\n")
        continue
    n_flagged = (df_model[flag] == 1).sum()
    print(f"{flag}:")
    print(f"  PLR flagged == 1: {n_flagged}")
    print(f"  Distinct values: {df_model[flag].nunique()}")
    print(df_model[flag].value_counts(dropna=False).to_string())
    print()

re_baualter_unsicher:
  PLR flagged == 1: 0
  Distinct values: 1
re_baualter_unsicher
0.0    535

re_brw_trend_outlier:
  PLR flagged == 1: 0
  Distinct values: 1
re_brw_trend_outlier
0    535

re_miet_dimension_anwendbar: already dropped



In [18]:
#decision: drop them as they are a constant after dropping 8 PLRs

df_model = df_model.drop(columns=[c for c in
    ["re_baualter_unsicher", "re_brw_trend_outlier", "re_miet_dimension_anwendbar"]
    if c in df_model.columns])

#Correlation Matrix at the end to check for correlations with all variables now in the dataset 

In [16]:
#build numeric column list from the CURRENT modelling frame
id_label = ["plr_id", "plr_name", "bez"]
num_cols = [c for c in df_model.columns if c not in id_label]

#drop helper columns and the remaining flag
cols_to_use = [
    c for c in num_cols
    if not c.endswith("_unsicher") and not c.startswith("com_has_")
]

#Spearman correlation matrix
corr = df_model[cols_to_use].corr(method="spearman")

#every unique pair once
results = []
for i in range(len(cols_to_use)):
    for j in range(i + 1, len(cols_to_use)):       # j > i: no duplicates/self-pairs
        var_a, var_b = cols_to_use[i], cols_to_use[j]
        rho = corr.loc[var_a, var_b]
        dim_a = var_a.split("_")[0]                 # "re" / "soc" / "com"
        dim_b = var_b.split("_")[0]
        results.append({
            "var_a": var_a,
            "var_b": var_b,
            "rho": rho,
            "abs_rho": abs(rho),
            "cross_dim": dim_a != dim_b,
        })

pairs = pd.DataFrame(results)

#redundancy-suspect pairs: |rho| > 0.95
high = pairs[pairs["abs_rho"] > 0.95].sort_values("abs_rho", ascending=False)
print(f"Variable pairs with |rho| > 0.95: {len(high)}")
print(high[["var_a", "var_b", "rho", "cross_dim"]].to_string(index=False))

#early warning: strong cross-dimension pairs |rho| > 0.8
cross_strong = pairs[(pairs["cross_dim"]) & (pairs["abs_rho"] > 0.8)].sort_values("abs_rho", ascending=False)
print(f"\nCross-dimension pairs with |rho| > 0.8: {len(cross_strong)}")
print(cross_strong[["var_a", "var_b", "rho"]].to_string(index=False))

Variable pairs with |rho| > 0.95: 0
Empty DataFrame
Columns: [var_a, var_b, rho, cross_dim]
Index: []

Cross-dimension pairs with |rho| > 0.8: 0
Empty DataFrame
Columns: [var_a, var_b, rho]
Index: []


In [19]:
df_model.columns

Index(['plr_id', 'plr_name', 'bez', 're_miete_niveau', 're_miete_trend',
       're_miete_unsicher', 're_leerstandsquote', 're_brw_niveau',
       're_brw_trend', 're_cov_wohnen_2025', 're_dichte_all',
       'soc_young_to_middle_adult_share_2025',
       'soc_young_to_middle_adult_share_change_2021_2025',
       'soc_single_person_hh_share_2024',
       'soc_single_person_hh_share_change_2021_2024',
       'soc_households_without_minor_children_share_2024',
       'soc_households_without_minor_children_share_change_2021_2024',
       'soc_internal_migration_volume_rate_2025',
       'soc_internal_migration_volume_rate_change_2021_2025',
       'soc_internal_net_migration_rate_2025',
       'soc_internal_net_migration_rate_change_2021_2025',
       'soc_transfer_benefit_share_2024',
       'soc_transfer_benefit_share_change_2020_2024',
       'soc_single_parent_household_share_2024',
       'soc_single_parent_household_share_change_2021_2024',
       'com_median_age_level_2026', 'com_s